# Cuaderno 04: Benchmark Comparativo de Modelos y Explicabilidad SHAP

## 1. Diseno Experimental del Benchmark y Validacion Cruzada
El protocolo de modelado en `src/modelos/evaluador_metricas.py` garantiza comparabilidad estricta sin fuga de datos:
- **Particion estratificada por cuantiles (70 / 15 / 15):** Sobre las 800 pistas limpias de `respaldo_tiktok_csv` (`MIN_VIEWS >= 100`), se asignan `560` registros a entrenamiento, `120` a validacion y `120` a prueba (`SEMILLA_ALEATORIA = 42`, `BINS_ESTRATIFICACION = 5`).
- **Preprocesamiento ajustado solo en entrenamiento:** Cotas de Winsorization IQR (`factor=1.5`, percentiles `1%` y `99%`), imputacion por mediana y `StandardScaler` se estiman unicamente sobre los 560 registros de `train`.
- **Familias evaluadas:** Regresores lineales regularizados (`RidgeCV`, `ElasticNetCV`), perceptron multicapa (`MLPRegressor` con arquitectura `128-64-32` y parada temprana), `RandomForestRegressor` (200 arboles) y `XGBRegressor` optimizado por `GridSearchCV` con **validacion cruzada de 5 pliegues** (`18` combinaciones de `max_depth`, `learning_rate` y `n_estimators`).


In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score

raiz = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from config import settings
from src.modelos import cargar_datos_entrenamiento, preparar_particiones_modelado
from src.modelos.modelo_arboles import crear_xgboost

reporte = json.loads(settings.RUTA_REPORTE_MODELOS.read_text(encoding='utf-8'))
df_benchmark = (
    pd.DataFrame.from_dict(reporte['metricas_test'], orient='index')
    .rename_axis('modelo')
    .reset_index()
    .sort_values('RMSE', ascending=True)
    .reset_index(drop=True)
)
print(
    f"Fuente: {reporte['fuente_datos']} | Particiones: "
    f"{reporte['n_train']}/{reporte['n_val']}/{reporte['n_test']}"
)
print(
    f"Mejor modelo en test: {reporte['mejor_modelo_test']} | "
    f"Seleccionado para produccion: {reporte['modelo_seleccionado_produccion']}"
)
print('\nTabla comparativa en prueba (ordenada por RMSE):')
print(df_benchmark.round(6).to_string(index=False))

df_crudo, _ = cargar_datos_entrenamiento()
particiones = preparar_particiones_modelado(df_crudo, guardar_escalador=False)
cols = settings.COLUMNAS_ACUSTICAS
x_train = particiones['df_train_esc'][cols]
y_train = particiones['df_train_esc'][settings.COLUMNA_TARGET_LOG].to_numpy(dtype=float)
x_test = particiones['df_test_esc'][cols]
y_test = particiones['df_test_esc'][settings.COLUMNA_TARGET_LOG].to_numpy(dtype=float)

cv_info = reporte['optimizacion_cv']
params_xgb = cv_info['mejores_parametros']
xgb_cv = crear_xgboost(
    profundidad_max=params_xgb['max_depth'],
    tasa_aprendizaje=params_xgb['learning_rate'],
    n_estimadores=params_xgb['n_estimators'],
)
kf = KFold(n_splits=settings.PLIEGUES_CV, shuffle=True, random_state=settings.SEMILLA_ALEATORIA)
rmse_pliegues = -cross_val_score(
    xgb_cv, x_train, y_train, cv=kf, scoring='neg_root_mean_squared_error'
)
print(
    f"\nValidacion cruzada {cv_info['pliegues_cv']}-fold XGBoost "
    f"({cv_info['total_combinaciones']} combinaciones):"
)
print(f'Mejores hiperparametros: {params_xgb}')
print('RMSE por pliegue:', [round(float(v), 6) for v in rmse_pliegues])
print(
    f'RMSE CV media: {rmse_pliegues.mean():.6f} +- {rmse_pliegues.std():.6f} '
    f"(reporte JSON: {cv_info['mejor_rmse_cv']:.6f})"
)

modelo_xgb = joblib.load(settings.RUTA_MODELO_XGBOOST)
pred_xgb = modelo_xgb.predict(x_test)
rng = np.random.default_rng(settings.SEMILLA_ALEATORIA)
boot_rmse = [
    root_mean_squared_error(y_test[idx], pred_xgb[idx])
    for idx in (rng.integers(0, len(y_test), size=len(y_test)) for _ in range(1000))
]
ic_inf, ic_sup = np.percentile(boot_rmse, [2.5, 97.5])
print(f'IC 95% bootstrap RMSE test (XGBoost, 1000 remuestreos): [{ic_inf:.6f}, {ic_sup:.6f}]')

fig_bench = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('RMSE en prueba (menor es mejor)', 'Coeficiente R2 en prueba (mayor es mejor)'),
)
colores_modelos = ['#22c55e' if m == 'xgboost' else '#6366f1' for m in df_benchmark['modelo']]
fig_bench.add_trace(
    go.Bar(
        x=df_benchmark['modelo'],
        y=df_benchmark['RMSE'],
        marker_color=colores_modelos,
        text=df_benchmark['RMSE'].round(4),
        textposition='outside',
    ),
    row=1,
    col=1,
)
fig_bench.add_trace(
    go.Bar(
        x=df_benchmark['modelo'],
        y=df_benchmark['R2'],
        marker_color=colores_modelos,
        text=df_benchmark['R2'].round(4),
        textposition='outside',
    ),
    row=1,
    col=2,
)
fig_bench.update_layout(
    title='Benchmark comparativo sobre conjunto de prueba (n=120)',
    height=440,
    showlegend=False,
)
fig_bench.update_yaxes(title_text='RMSE (escala ln(1+x))', range=[0, 0.0185], row=1, col=1)
fig_bench.update_yaxes(title_text='R2', range=[-0.72, 0.36], row=1, col=2)
fig_bench.show()


Fuente: respaldo_tiktok_csv | Particiones: 560/120/120
Mejor modelo en test: elasticnet | Seleccionado para produccion: xgboost

Tabla comparativa en prueba (ordenada por RMSE):
       modelo      MAE     RMSE        R2
   elasticnet 0.008573 0.010699  0.253548
random_forest 0.008848 0.010889  0.226803
        ridge 0.008691 0.010944  0.218940
      xgboost 0.008935 0.011012  0.209286
          mlp 0.012307 0.015471 -0.560757

Validacion cruzada 5-fold XGBoost (18 combinaciones):
Mejores hiperparametros: {'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 100}
RMSE por pliegue: [0.01137, 0.01078, 0.011875, 0.012158, 0.012028]
RMSE CV media: 0.011642 +- 0.000507 (reporte JSON: 0.011642)
IC 95% bootstrap RMSE test (XGBoost, 1000 remuestreos): [0.009813, 0.012079]


<plotly.graph_objs.Figure>

### Analisis de Brecha de Desempeno entre Modelos Lineales, MLP y Ensambles
- **Paridad estadistica en el bloque superior:** `ElasticNet` obtiene el menor error puntual en prueba (`RMSE = 0.010699`, `MAE = 0.008573`, `R2 = 0.2535`), seguido muy de cerca por `RandomForest` (`RMSE = 0.010889`, `R2 = 0.2268`), `Ridge` (`RMSE = 0.010944`, `R2 = 0.2189`) y `XGBoost` (`RMSE = 0.011012`, `MAE = 0.008935`, `R2 = 0.2093`). La diferencia entre `ElasticNet` y `XGBoost` es de apenas `0.000313` en escala `ln(1+x)`, y los cuatro modelos caen comodamente dentro del intervalo de confianza bootstrap al 95 % de `XGBoost` (`[0.009813, 0.012079]`), confirmando paridad estadistica en $n_{\text{test}} = 120$.
- **Por que destacan ElasticNet y XGBoost regularizado:** En `respaldo_tiktok_csv`, la senal latente posee una componente cuasi-lineal dominante en su zona central combinada con colinealidad entre bandas espectrales, favoreciendo la seleccion L1/L2 de `ElasticNet`. A su vez, `GridSearchCV` (5 pliegues) selecciona la configuracion mas conservadora para `XGBoost` (`max_depth = 3`, `learning_rate = 0.03`, `n_estimators = 100`), logrando una desviacion entre pliegues muy baja (`RMSE CV = 0.011642 +- 0.000507`).
- **Colapso del MLPRegressor (`R2 = -0.5608`):** La arquitectura densa `(128, 64, 32)` cuenta con miles de parametros entrenados sobre ~476 muestras efectivas (tras descontar el 15 % interno para `early_stopping`), sufriendo sobreajuste severo frente a los metodos regularizados y de ensamble.


In [2]:
from src.modelos import (
    calcular_importancia_global_shap,
    explicar_prediccion_individual,
)

umbrales = json.loads(settings.RUTA_UMBRALES_VIRALES.read_text(encoding='utf-8'))
monotonia_ok = all(u['p25'] <= u['p50'] <= u['p75'] for u in umbrales.values())
print(f'Variables en umbrales_virales.json: {len(umbrales)} | Monotonia p25<=p50<=p75: {monotonia_ok}')
vars_clave = [
    'tempo_bpm',
    'initial_silence_sec',
    'rms_mean',
    'onset_strength_mean',
    'spectral_centroid',
    'beat_regularity',
]
df_umbrales_clave = pd.DataFrame([{'variable': v, **umbrales[v]} for v in vars_clave])
print('\nMuestra de umbrales virales (top 15% entrenamiento en escala fisica original):')
print(df_umbrales_clave.round(4).to_string(index=False))

importancia_global = calcular_importancia_global_shap(modelo_xgb, x_test)
print('\nTop 10 atributos por importancia global SHAP (mean |phi| en prueba):')
print(importancia_global.head(10).round(6).to_string(index=False))

muestra_local = x_test.iloc[[0]]
pred_modelo = float(modelo_xgb.predict(muestra_local)[0])
explicacion_local = explicar_prediccion_individual(modelo_xgb, muestra_local)
suma_aditiva = explicacion_local['valor_base'] + sum(explicacion_local['contribuciones'].values())
error_aditividad = abs(pred_modelo - suma_aditiva)

print('\nAuditoria de aditividad local TreeExplainer (pista test[0]):')
print(f"  Valor base E[f(X)] : {explicacion_local['valor_base']:.8f}")
print(f'  Suma base + SHAP   : {suma_aditiva:.8f}')
print(f'  Prediccion XGBoost : {pred_modelo:.8f}')
print(f'  Error absoluto     : {error_aditividad:.2e}')
print('  Atributos favorables (top 3):', explicacion_local['atributos_favorables'])
print('  Atributos perjudiciales (top 3):', explicacion_local['atributos_perjudiciales'])

top_global = importancia_global.head(12).sort_values('importancia_media_abs', ascending=True)
top_local = sorted(
    explicacion_local['contribuciones'].items(),
    key=lambda kv: abs(kv[1]),
    reverse=True,
)[:12]
top_local = sorted(top_local, key=lambda kv: kv[1])

fig_shap = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Importancia global SHAP (top 12 en prueba)', 'Atribucion local SHAP (pista test[0])'),
    horizontal_spacing=0.24,
)
fig_shap.add_trace(
    go.Bar(x=top_global['importancia_media_abs'], y=top_global['variable'], orientation='h', marker_color='#8b5cf6'),
    row=1,
    col=1,
)
fig_shap.add_trace(
    go.Bar(
        x=[v for _, v in top_local],
        y=[k for k, _ in top_local],
        orientation='h',
        marker_color=['#22c55e' if v >= 0 else '#ef4444' for _, v in top_local],
    ),
    row=1,
    col=2,
)
fig_shap.update_layout(
    title='Explicabilidad matematica TreeExplainer sobre XGBoost',
    height=480,
    showlegend=False,
    margin=dict(l=160, r=30, t=65, b=45),
)
fig_shap.update_xaxes(title_text='Media |SHAP|', row=1, col=1)
fig_shap.update_xaxes(title_text='Contribucion SHAP (phi_j)', row=1, col=2)
fig_shap.show()


Variables en umbrales_virales.json: 28 | Monotonia p25<=p50<=p75: True

Muestra de umbrales virales (top 15% entrenamiento en escala fisica original):
           variable       p25       p50       p75     media
          tempo_bpm  111.8275  126.0160  135.2708  123.5125
initial_silence_sec    0.2228    0.2569    0.3383    0.2743
           rms_mean    0.2035    0.2397    0.2626    0.2326
onset_strength_mean    4.6509    5.1374    5.4371    5.0552
  spectral_centroid 2951.9200 3242.4800 3535.7900 3227.4060
    beat_regularity    0.7991    0.8553    0.8935    0.8390

Top 10 atributos por importancia global SHAP (mean |phi| en prueba):
           variable  importancia_media_abs  impacto_medio_con_signo
initial_silence_sec               0.002363                 0.000438
onset_strength_mean               0.001603                -0.000362
          tempo_bpm               0.001360                -0.000247
             mfcc_1               0.001072                -0.000252
            rms_std

<plotly.graph_objs.Figure>

## 2. Conclusion Ejecutiva: Seleccion de XGBoost y Explicabilidad en Produccion
1. **Criterio integral para desplegar XGBoost:** Aunque `ElasticNet` registro un `RMSE` ligeramente menor en la muestra de prueba (`0.010699` frente a `0.011012`), la brecha de `0.000313` en escala `ln(1+x)` es estadisticamente indistinguible dentro del intervalo bootstrap (`[0.009813, 0.012079]`) y equivale a menos de `0.04` puntos porcentuales en engagement. `XGBoost` se selecciona para produccion en `app/servicios/orquestador_inferencia.py` porque captura relaciones no lineales esenciales en audio DSP real —destacando `tempo_bpm`, que ocupa el tercer lugar en importancia global SHAP (`0.001360`) pese a su correlacion de Spearman casi nula (`+0.0425`)— junto con `initial_silence_sec` (`0.002363`), `onset_strength_mean` (`0.001603`), `mfcc_1` (`0.001072`) y `rms_std` (`0.000865`).
2. **Exactitud de Shapley en tiempo polinomial y anclaje fisico:** `shap.TreeExplainer` verifica la propiedad de aditividad local $\hat{f}(x) = \phi_0 + \sum_{j=1}^{28} \phi_j(x)$ con un error absoluto de apenas `1.15e-07` (`float32`), ejecutandose en milisegundos sin muestreo Monte Carlo. Combinado con `modelos_guardados/umbrales_virales.json` (28 variables del top 15 % viral en unidades fisicas reales, `p25 <= p50 <= p75` verificado, p. ej. `tempo_bpm` mediana `126.02 BPM` e `initial_silence_sec` mediana `0.2569 s`), dota al detector de deficiencias y al agente Ollama de evidencia cuantitativa auditable.
